In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split  # for train-test splitting
from sklearn.metrics import (accuracy_score,
                            classification_report,
                            confusion_matrix,
                            f1_score,ConfusionMatrixDisplay)
from collections.abc import Callable
from typing import Literal
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary

In [2]:
###----------------------
### Some basic parameters
###----------------------

inpDir = Path('..') / '..' / 'input'
outDir = Path('..') / 'output'
modelDir = Path('..') / 'models'
subDir = 'fifa_2019'

RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(seed = RANDOM_STATE) # Set Random Seed for reproducible  results

EPOCHS = 101 # number of epochs
BATCH_SIZE = 32
ALPHA = 0.001 # learning rate
TEST_SIZE = 0.2
TRAIN_SIZE=453*BATCH_SIZE

# parameters for Matplotlib
params = {'legend.fontsize': 'medium',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'medium',
          'axes.titlesize':'large',
          'xtick.labelsize':'medium',
          'ytick.labelsize':'medium'
         }

plt.rcParams.update(params)

CMAP = plt.cm.coolwarm
plt.style.use('seaborn-v0_8-darkgrid') # plt.style.use('ggplot')

In [3]:
# Check if all directories are present
outDir.mkdir(parents=True, exist_ok=True)

modelSubDir = modelDir/ subDir
modelSubDir.mkdir(parents=True, exist_ok=True)

In [4]:
data_df = pd.read_csv('fifa_2019.csv')
data_df.columns

Index(['Unnamed: 0', 'ID', 'Name', 'Age', 'Photo', 'Nationality', 'Flag',
       'Overall', 'Potential', 'Club', 'Club Logo', 'Value', 'Wage', 'Special',
       'Preferred Foot', 'International Reputation', 'Weak Foot',
       'Skill Moves', 'Work Rate', 'Body Type', 'Real Face', 'Position',
       'Jersey Number', 'Joined', 'Loaned From', 'Contract Valid Until',
       'Height', 'Weight', 'LS', 'ST', 'RS', 'LW', 'LF', 'CF', 'RF', 'RW',
       'LAM', 'CAM', 'RAM', 'LM', 'LCM', 'CM', 'RCM', 'RM', 'LWB', 'LDM',
       'CDM', 'RDM', 'RWB', 'LB', 'LCB', 'CB', 'RCB', 'RB', 'Crossing',
       'Finishing', 'HeadingAccuracy', 'ShortPassing', 'Volleys', 'Dribbling',
       'Curve', 'FKAccuracy', 'LongPassing', 'BallControl', 'Acceleration',
       'SprintSpeed', 'Agility', 'Reactions', 'Balance', 'ShotPower',
       'Jumping', 'Stamina', 'Strength', 'LongShots', 'Aggression',
       'Interceptions', 'Positioning', 'Vision', 'Penalties', 'Composure',
       'Marking', 'StandingTackle', 'SlidingT

In [5]:
data_df = data_df[data_df["Position"].notnull()]
data_df.head()

,Unnamed: 0,ID,Name,Age,Photo,Nationality,Flag,Overall,Potential,Club,...,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Release Clause
0,0,158023,L. Messi,31,https://cdn.sofifa.org/players/4/19/158023.png,Argentina,https://cdn.sofifa.org/flags/52.png,94,94,FC Barcelona,...,96.0,33.0,28.0,26.0,6.0,11.0,15.0,14.0,8.0,€226.5M
1,1,20801,Cristiano Ronaldo,33,https://cdn.sofifa.org/players/4/19/20801.png,Portugal,https://cdn.sofifa.org/flags/38.png,94,94,Juventus,...,95.0,28.0,31.0,23.0,7.0,11.0,15.0,14.0,11.0,€127.1M
2,2,190871,Neymar Jr,26,https://cdn.sofifa.org/players/4/19/190871.png,Brazil,https://cdn.sofifa.org/flags/54.png,92,93,Paris Saint-Germain,...,94.0,27.0,24.0,33.0,9.0,9.0,15.0,15.0,11.0,€228.1M
3,3,193080,De Gea,27,https://cdn.sofifa.org/players/4/19/193080.png,Spain,https://cdn.sofifa.org/flags/45.png,91,93,Manchester United,...,68.0,15.0,21.0,13.0,90.0,85.0,87.0,88.0,94.0,€138.6M
4,4,192985,K. De Bruyne,27,https://cdn.sofifa.org/players/4/19/192985.png,Belgium,https://cdn.sofifa.org/flags/7.png,91,92,Manchester City,...,88.0,68.0,58.0,51.0,15.0,13.0,5.0,10.0,13.0,€196.4M


In [6]:
rel_cols = ["Position", 'Finishing', 'HeadingAccuracy', 'ShortPassing', 'Volleys', 'Dribbling',
            'Curve', 'FKAccuracy', 'LongPassing', 'BallControl', 'Acceleration',
            'SprintSpeed', 'Agility', 'Reactions', 'Balance', 'ShotPower',
            'Jumping', 'Stamina', 'Strength', 'LongShots', 'Aggression',
            'Interceptions', 'Positioning', 'Vision', 'Penalties', 'Composure',
            'Marking', 'StandingTackle', 'SlidingTackle', 'GKDiving', 'GKHandling',
            'GKKicking', 'GKPositioning', 'GKReflexes']

In [7]:
goalkeeper = 'GK'
forward = ['ST', 'LW', 'RW', 'LF', 'RF', 'RS','LS', 'CF']
midfielder = ['CM','RCM','LCM', 'CDM','RDM','LDM', 'CAM', 'LAM', 'RAM', 'RM', 'LM']
defender = ['CB', 'RCB', 'LCB', 'LWB', 'RWB', 'LB', 'RB']

In [8]:
data_df=data_df[rel_cols]
data_df.head()

,Position,Finishing,HeadingAccuracy,ShortPassing,Volleys,Dribbling,Curve,FKAccuracy,LongPassing,BallControl,...,Penalties,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes
0,RF,95.0,70.0,90.0,86.0,97.0,93.0,94.0,87.0,96.0,...,75.0,96.0,33.0,28.0,26.0,6.0,11.0,15.0,14.0,8.0
1,ST,94.0,89.0,81.0,87.0,88.0,81.0,76.0,77.0,94.0,...,85.0,95.0,28.0,31.0,23.0,7.0,11.0,15.0,14.0,11.0
2,LW,87.0,62.0,84.0,84.0,96.0,88.0,87.0,78.0,95.0,...,81.0,94.0,27.0,24.0,33.0,9.0,9.0,15.0,15.0,11.0
3,GK,13.0,21.0,50.0,13.0,18.0,21.0,19.0,51.0,42.0,...,40.0,68.0,15.0,21.0,13.0,90.0,85.0,87.0,88.0,94.0
4,RCM,82.0,55.0,92.0,82.0,86.0,85.0,83.0,91.0,91.0,...,79.0,88.0,68.0,58.0,51.0,15.0,13.0,5.0,10.0,13.0


In [9]:
#Assign labels to goalkeepers
data_df['Position_encoded']=0
data_df.loc[data_df["Position"] == "GK", "Position_encoded"] = 0

#Defenders
data_df.loc[data_df["Position"].isin(defender), "Position_encoded"] = 1

#Midfielders
data_df.loc[data_df["Position"].isin(midfielder), "Position_encoded"] = 2

#Forward
data_df.loc[data_df["Position"].isin(forward), "Position_encoded"] = 3

# Convert Column "Position" to numeric so that Pandas does not complain
data_df['Position_encoded'] = pd.to_numeric(data_df['Position_encoded'], downcast="integer")

In [10]:
labels=data_df['Position_encoded']
features=data_df.drop(['Position','Position_encoded'],axis=1)

In [11]:
class_names = {0: 'Goal Keeper', 1: 'Defender', 2: 'Mid-Fielder', 3: 'Forward'}

In [12]:
# splitting in train ans test datasets
X_train, X_test,y_train, y_test = train_test_split(features,labels,
                                     stratify=labels,
                                     test_size=TEST_SIZE, 
                                     random_state=RANDOM_STATE )
X_train.shape, X_test.shape,y_train.shape, y_test.shape

((14517, 33), (3630, 33), (14517,), (3630,))

In [13]:
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.fit_transform(X_test)

In [14]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [15]:
class FifaDataset(Dataset):
    def __init__(self,X,y):
        super().__init__()
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)

    def __len__(self):
        return len(self.X)
    def __getitem__(self,index):
        return self.X[index], self.y[index]

In [16]:
train_dataset=FifaDataset(X_train,y_train)
train_loader=DataLoader(dataset=train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [17]:
for batch_idx, (data,target) in enumerate(train_loader):
    print(f'Batch: {batch_idx +1}: ', end=' ')
    print(f'data: {data.shape}: ', end=' ')
    print(f'Target: {target.shape}: ')

Batch: 1:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 2:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 3:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 4:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 5:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 6:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 7:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 8:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 9:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 10:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 11:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 12:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 13:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 14:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 15:  data: torch.Size([32, 33]):  Target: torch.Si

In [18]:
X_train.shape, X_test.shape


((14517, 33), (3630, 33))

In [19]:
test_dataset=FifaDataset(X_test,y_test.to_numpy())
test_loader=DataLoader(dataset=train_dataset,batch_size=BATCH_SIZE, shuffle=True)

In [20]:
for batch_idx, (data,target) in enumerate(test_loader):
    print(f'Batch: {batch_idx +1}: ', end=' ')
    print(f'data: {data.shape}: ', end=' ')
    print(f'Target: {target.shape}: ')

Batch: 1:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 2:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 3:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 4:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 5:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 6:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 7:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 8:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 9:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 10:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 11:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 12:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 13:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 14:  data: torch.Size([32, 33]):  Target: torch.Size([32]): 
Batch: 15:  data: torch.Size([32, 33]):  Target: torch.Si

In [22]:
class FifaModel(nn.Module):

    '''Layers:
    33  -> 16 -> 8-> 4
    
    '''
    def __init__(self,input_dim):
        super(FifaModel,self).__init__()
        self.layer1=nn.Linear(input_dim,16)
        self.active1=nn.ReLU()
        
        self.layer2=nn.Linear(16,8)
        self.active2=nn.ReLU()

        self.layer3=nn.Linear(8,4)
    
    def forward(self,x):
        x=self.layer1(x)
        x=self.active1(x)
        x=self.layer2(x)
        x=self.active2(x)
        x=self.layer3(x)
        return x

model=FifaModel(input_dim=X_train.shape[1]).to(device=device)

In [ ]:
# for features, labels

In [23]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=ALPHA)

loss,tloss,n_epoch,acc,tacc=[],[],[],[],[]
for epoch in range(EPOCHS):
    model.train()
    epoch_loss=0
    epoch_acc=0
    tepoch_loss=0
    tepoch_acc=0

    for batch_idx,(train_X,train_y) in enumerate(train_loader):
        train_X,train_y=train_X.to(device),train_y.to(device)
        predict_prob=model(train_X)
        batch_loss=loss_fn(predict_prob,train_y)
        epoch_loss+=(batch_loss-epoch_loss)/(batch_idx+1)

        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        _, y_pred=torch.max(predict_prob,1)
        batch_acc=accuracy_score(train_y.cpu().numpy(),
                                y_pred.data.cpu())